# NLBSE'23 — Dataset Feasibility

## 1. Dataset Overview & Schema

This notebook explores the official NLBSE'23 training dataset to assess its
structure, label distribution, missing values, and processing feasibility.

The official test set is reserved for final evaluation.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/raw/nlbse23-issue-classification-train.csv")

CHUNK_SIZE = 50_000

In [2]:
sample = pd.read_csv(DATA_PATH, nrows=5)

print("Shape:", sample.shape)
print("\nColumns:", sample.columns.tolist())
print("\nDtypes:")
print(sample.dtypes)

display(sample.head())

Shape: (5, 5)

Columns: ['id', 'labels', 'title', 'body', 'author_association']

Dtypes:
id                    int64
labels                  str
title                   str
body                    str
author_association      str
dtype: object


,id,labels,title,body,author_association
0,1199051804,documentation,setting a logging Handler name,BPO | [43058](https://bugs.python.org/issue430...,MANNEQUIN
1,1199074324,documentation,Improve documentation for typing._GenericAlias,BPO | [46589](https://bugs.python.org/issue465...,MANNEQUIN
2,1199022454,documentation,Description of '\w' behavior is vague in `re` ...,BPO | [38566](https://bugs.python.org/issue385...,MANNEQUIN
3,1199028356,documentation,add docstrings to functions in pdb module,BPO | [39278](https://bugs.python.org/issue392...,MANNEQUIN
4,1199055394,documentation,Documentation needs to declare CalledProcessEr...,BPO | [43635](https://bugs.python.org/issue436...,MANNEQUIN


In [3]:
from collections import Counter

label_counts = Counter()
total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["labels"],
    chunksize=CHUNK_SIZE,
):
    total_rows += len(chunk)
    label_counts.update(chunk["labels"].dropna())

print("Total rows:", total_rows)
print("\nLabel distribution:")

display(
    pd.Series(label_counts)
    .sort_values(ascending=False)
    .rename("count")
    .to_frame()
)

Total rows: 1275881

Label distribution:


,count
bug,670951
feature,472216
question,76048
documentation,56666


In [4]:
missing_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE):
    missing_counts = missing_counts.add(
        chunk.isna().sum(),
        fill_value=0,
    )

missing_counts = missing_counts.astype("int64")

display(missing_counts.sort_values(ascending=False))

body                  4681
title                    4
author_association       0
id                       0
labels                   0
dtype: int64

## 2. Text Characteristics & Dataset Artifacts

Analyze text lengths, common formatting patterns, and potential
dataset-specific artifacts before making preprocessing decisions.

In [5]:
title_lengths = []
body_lengths = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["title", "body"],
    chunksize=CHUNK_SIZE,
):
    title_lengths.extend(chunk["title"].fillna("").str.len())
    body_lengths.extend(chunk["body"].fillna("").str.len())

lengths = pd.DataFrame(
    {
        "title_length": title_lengths,
        "body_length": body_lengths,
    }
)

display(
    lengths.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

,title_length,body_length
count,1.275881e+06,1.275881e+06
mean,4.776691e+01,1.208326e+03
std,2.491692e+01,3.993738e+03
min,0.000000e+00,0.000000e+00
50%,4.400000e+01,4.900000e+02
75%,6.100000e+01,1.076000e+03
90%,7.900000e+01,2.272000e+03
95%,9.200000e+01,3.835000e+03
99%,1.230000e+02,1.235940e+04
max,9.940000e+02,2.591590e+05


In [6]:
patterns = {
    "markdown_links": r"\]\(https?://",
    "html_tags": r"<[a-zA-Z][^>]*>",
    "code_blocks": r"```",
    "github_mentions": r"@[A-Za-z0-9_-]+",
    "bpo_reference": r"bugs\.python\.org",
    "stack_trace": r"Traceback \(most recent call last\)",
}

pattern_counts = dict.fromkeys(patterns, 0)

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["body"],
    chunksize=CHUNK_SIZE,
):
    bodies = chunk["body"].fillna("")

    for name, pattern in patterns.items():
        pattern_counts[name] += bodies.str.contains(
            pattern,
            regex=True,
            case=False,
        ).sum()

pattern_results = pd.DataFrame(
    {
        "count": pattern_counts,
    }
)

pattern_results["percentage"] = (
    pattern_results["count"] / total_rows * 100
).round(2)

display(pattern_results.sort_values("count", ascending=False))

,count,percentage
markdown_links,336860,26.40
code_blocks,301684,23.65
html_tags,160593,12.59
github_mentions,121753,9.54
stack_trace,21258,1.67
bpo_reference,236,0.02


In [7]:
samples = []

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
):
    samples.append(
        chunk.groupby("labels", group_keys=False)
        .sample(frac=0.002, random_state=42)
    )

sampled_data = pd.concat(samples, ignore_index=True)

display(
    sampled_data.groupby("labels", group_keys=False)
    .sample(n=3, random_state=42)[
        ["labels", "title", "body", "author_association"]
    ]
)

,labels,title,body,author_association
1533,bug,No I or D terms,"I have tried to run this algorithm on many, ma...",NONE
1021,bug,Delta mass plot labelling wrong,"In the beginning, ""count"" is selected but ""fre...",COLLABORATOR
648,bug,[Bug]: Incompatible with Nova 4.0,### Is the bug applicable and reproducable to ...,NONE
258,documentation,[Docs] unidirectional async section command sy...,In the Setting Up Unidirectional Replication s...,CONTRIBUTOR
1352,documentation,Documentation takes a very long time to build ...,Either reduce timing for notebooks or exclude ...,MEMBER
155,documentation,How accelerators work with regards to what que...,How accelerators work with regards to what que...,COLLABORATOR
1164,feature,Add the ability to specify the RDS & S3 secret...,**Is your feature request related to a problem...,CONTRIBUTOR
570,feature,update the SDK version,Following the Rebase onto vector-im/element-io...,COLLABORATOR
271,feature,Isomorphism in SBML-multi export,In order to export BCSL models to SBML-multi s...,COLLABORATOR
796,question,ISSUES 2,I'm having a problem with this.,OWNER


## 3. Duplicates & Metadata Analysis

Investigate duplicate issue IDs, repeated text, and the relationship
between author association and issue labels.

In [ ]:
import hashlib

seen_ids = set()
seen_texts = set()

duplicate_ids = 0
duplicate_texts = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["id", "title", "body"],
    chunksize=CHUNK_SIZE,
):
    chunk[["title", "body"]] = chunk[["title", "body"]].fillna("")

    for issue_id, title, body in chunk.itertuples(index=False, name=None):
        if issue_id in seen_ids:
            duplicate_ids += 1
        else:
            seen_ids.add(issue_id)

        text = f"{title or ''}\0{body or ''}"
        text_hash = hashlib.sha256(text.encode("utf-8")).digest()

        if text_hash in seen_texts:
            duplicate_texts += 1
        else:
            seen_texts.add(text_hash)

print("Duplicate IDs:", duplicate_ids)
print("Duplicate title + body:", duplicate_texts)

Duplicate IDs: 33353
Duplicate title + body: 53459


In [10]:
association_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["author_association"],
    chunksize=CHUNK_SIZE,
):
    counts = chunk["author_association"].value_counts()

    association_counts = association_counts.add(
        counts,
        fill_value=0,
    )

association_counts = association_counts.astype("int64")

display(association_counts.sort_values(ascending=False))

author_association
NONE            539679
CONTRIBUTOR     238469
OWNER           204397
MEMBER          165619
COLLABORATOR    127563
MANNEQUIN          154
dtype: int64

In [11]:
association_label_counts = pd.DataFrame()

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["author_association", "labels"],
    chunksize=CHUNK_SIZE,
):
    counts = pd.crosstab(
        chunk["author_association"],
        chunk["labels"],
    )

    association_label_counts = association_label_counts.add(
        counts,
        fill_value=0,
    )

association_label_counts = association_label_counts.fillna(0).astype("int64")

display(association_label_counts)

display(
    association_label_counts.div(
        association_label_counts.sum(axis=1),
        axis=0,
    ).round(3)
)

labels,bug,documentation,feature,question
author_association,,,,
COLLABORATOR,62683,8146,54827,1907
CONTRIBUTOR,129701,12840,90916,5012
MANNEQUIN,15,136,3,0
MEMBER,79335,8698,75588,1998
NONE,334879,14294,125072,65434
OWNER,64338,12552,125810,1697


labels,bug,documentation,feature,question
author_association,,,,
COLLABORATOR,0.491,0.064,0.430,0.015
CONTRIBUTOR,0.544,0.054,0.381,0.021
MANNEQUIN,0.097,0.883,0.019,0.000
MEMBER,0.479,0.053,0.456,0.012
NONE,0.621,0.026,0.232,0.121
OWNER,0.315,0.061,0.616,0.008


## 4. Duplicate Consistency & Label Conflicts

In [12]:
text_labels = {}
id_records = {}

text_label_conflicts = 0
id_label_conflicts = 0
id_text_conflicts = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["id", "title", "body", "labels"],
    chunksize=CHUNK_SIZE,
):
    chunk[["title", "body"]] = chunk[["title", "body"]].fillna("")

    for issue_id, title, body, label in chunk.itertuples(
        index=False, name=None
    ):
        text_hash = hashlib.sha256(
            f"{title}\0{body}".encode("utf-8")
        ).digest()

        if text_hash in text_labels:
            if text_labels[text_hash] != label:
                text_label_conflicts += 1
        else:
            text_labels[text_hash] = label

        if issue_id in id_records:
            previous_hash, previous_label = id_records[issue_id]

            if previous_label != label:
                id_label_conflicts += 1

            if previous_hash != text_hash:
                id_text_conflicts += 1
        else:
            id_records[issue_id] = (text_hash, label)

print("Text label conflict occurrences:", text_label_conflicts)
print("ID label conflict occurrences:", id_label_conflicts)
print("ID text conflict occurrences:", id_text_conflicts)

Text label conflict occurrences: 45643
ID label conflict occurrences: 2059
ID text conflict occurrences: 1714


In [15]:
from collections import Counter, defaultdict

text_groups = defaultdict(Counter)

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["title", "body", "labels"],
    chunksize=CHUNK_SIZE,
):
    chunk[["title", "body"]] = chunk[["title", "body"]].fillna("")

    for label, title, body in chunk[
        ["labels", "title", "body"]
    ].itertuples(index=False, name=None):

        text_hash = hashlib.sha256(
            f"{title}\0{body}".encode("utf-8")
        ).digest()

        text_groups[text_hash][label] += 1

conflicting_groups = {
    key: counts
    for key, counts in text_groups.items()
    if len(counts) > 1
}

print("Unique text groups:", len(text_groups))
print("Conflicting text groups:", len(conflicting_groups))
print(
    "Rows in conflicting groups:",
    sum(sum(counts.values()) for counts in conflicting_groups.values()),
)

print("\nTop conflicting label combinations:")

label_combinations = Counter(
    tuple(sorted(counts))
    for counts in conflicting_groups.values()
)

print(label_combinations.most_common(10))

Unique text groups: 1222422
Conflicting text groups: 440
Rows in conflicting groups: 4146

Top conflicting label combinations:
[(('bug', 'feature'), 133), (('documentation', 'feature'), 113), (('bug', 'question'), 105), (('bug', 'documentation'), 36), (('feature', 'question'), 34), (('documentation', 'question'), 9), (('bug', 'documentation', 'feature', 'question'), 4), (('documentation', 'feature', 'question'), 3), (('bug', 'documentation', 'feature'), 3)]


### Dataset Overview

- The official NLBSE'23 training dataset contains **1,275,881 issue reports** with five columns: `id`, `labels`, `title`, `body`, and `author_association`.
- The dataset contains four target classes: `bug`, `feature`, `question`, and `documentation`.
- The official test set remains untouched and is reserved for final evaluation.

### Class Distribution

- **Bug:** 670,951 (52.59%)
- **Feature:** 472,216 (37.01%)
- **Question:** 76,048 (5.96%)
- **Documentation:** 56,666 (4.44%)
- The dataset is imbalanced, with `bug` and `feature` accounting for nearly 90% of training examples.
- Evaluation should include **Micro F1, Macro F1, and per-class Precision, Recall, and F1**, rather than relying on accuracy alone.

### Missing Values

- `body`: 4,681 missing values.
- `title`: 4 missing values.
- No missing values were identified in `id`, `labels`, or `author_association`.
- Missing text fields should be handled explicitly during data preparation rather than automatically discarding otherwise usable issues.

### Text Characteristics

- Titles are relatively short, with a median length of **44 characters**.
- Bodies are substantially longer, with a median length of **490 characters**, a mean of approximately **1,208 characters**, and a maximum of **259,159 characters**.
- The long-tailed body-length distribution makes input truncation an important experiment, especially for transformer-based models.
- Planned input comparisons include **Title Only**, **Title + Body**, and **Title + Body with Truncation**.

### Text Structure and Preprocessing

- Approximately **26.40%** of issues contain Markdown links.
- Approximately **23.65%** contain fenced code blocks.
- Approximately **12.59%** contain HTML tags.
- Approximately **9.54%** contain GitHub mentions.
- These elements may contain useful classification signals, so aggressive text cleaning is not justified without experimental evidence.
- Future experiments will compare minimal preprocessing with structured transformations, including replacing fenced code blocks with a `CODE` placeholder.

### Duplicate and Label Consistency Analysis

- **1,222,422 unique title–body combinations** were identified.
- **53,459 duplicate text occurrences** were found, representing approximately **4.19%** of training rows.
- **440 unique text groups** contain conflicting labels, involving **4,146 rows**.
- Conflicting groups represent approximately **0.034% of unique text groups**.
- The most common conflicting label combinations were `bug`/`feature` (133 groups), `documentation`/`feature` (113), and `bug`/`question` (105).
- Duplicate IDs were also observed, but the exact semantics of the `id` field have not been established. Therefore, IDs will not be used as the sole criterion for removing records.
- The raw dataset will remain unchanged. Duplicate handling and conflict exclusion will be implemented in a reproducible preparation pipeline.

### Metadata Analysis

- The `author_association` field exhibits noticeable differences in label distribution across author roles.
- For example, `OWNER` issues are more frequently labeled `feature`, while issues from `NONE` authors are more frequently labeled `bug` or `question`.
- Although this metadata may be predictive, it could introduce shortcuts unrelated to issue text.
- The initial models will use text features only. Metadata may be revisited in a separate ablation experiment.

### Conclusions and Next Steps

- The dataset is sufficiently large and diverse to support meaningful comparisons between classical machine learning, neural networks, and transformer-based approaches.
- Class imbalance, duplicate text, conflicting labels, long issue bodies, and structured Markdown content are the main challenges identified during the feasibility audit.
- The next stage will implement reproducible data preparation and a leakage-aware, stratified training/validation split.
- The first modeling experiment will use **TF-IDF + Logistic Regression on issue titles only**, establishing a simple baseline before introducing additional text, preprocessing variants, or more complex models.
- All modeling decisions will be evaluated using training-derived validation data, keeping the official test set reserved for final evaluation.